# Демо 1 · DINOv2 vs DINOv3 на VisDrone: фічі без жодної мітки

**Заняття 5 · Vision Transformers та мультимодальні моделі** · слайди 17, 19, 22, 26–30, 46

Та сама архітектура ViT, навчена без міток (слайд 26). Перевіряємо на аерокадрах VisDrone, що це дає на практиці
і чим DINOv3 відрізняється від DINOv2:

| Блок | Що показуємо | Слайд |
|---|---|---|
| **1. Що бачить модель** | PCA патч-фіч, високонормні токени, увага [CLS]; DINOv2 → DINOv2 + регістри → DINOv3; patch 14 vs 16 | 17, 26–28 |
| **2. Клікни на ціль** | Подібність одного патча до всіх патчів інших відео: локалізація без детектора й без міток | 26, 22 |
| **3. 200 кадрів проти 2000** | kNN і linear probe на кропах об'єктів; випадковий добір міток проти кластерного | 30, 46 |
| **4. Архів як актив** | Карта сцен архіву, найсхожіші сцени з різних відео, витік даних при розбитті train/test | 25, 30 |

**Підготовка (один раз, з мережею):**
```bash
# з кореня репозиторію
python tools/fetch_dino_models.py                        # DINOv2, DINOv2-reg, DINOv3 (base) -> artifacts/models/dino/
python tools/fetch_visdrone.py --splits train val test-dev
```
DINOv3 на Hugging Face — **gated** (DINOv3 License, не Apache): прийміть умови на сторінці
[моделі](https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m), дочекайтесь підтвердження, `hf auth login`.
Без доступу ноутбук не падає: DINOv3 пропускається з підказкою, решта працює.

**Smoke-тест без мережі:** `DEMO1_SMOKE=1 jupyter nbconvert --to notebook --execute demo1_dino.ipynb --output smoke1.ipynb` —
синтетичні кадри й випадкові ваги, числа беззмістовні.

In [ ]:
import os, sys, json, time, subprocess
from pathlib import Path

SMOKE = os.environ.get("DEMO1_SMOKE") == "1"

# ---------------- Конфігурація ----------------
DEVICE = "cuda"            # "cuda" | "cpu" | None — авто. cuda без CUDA -> попередження і CPU
MODELS = ["dinov2", "dinov2reg", "dinov3"]   # + "dinov3sat" (лише SIZE="large"): DINOv3 на супутникових знімках SAT-493M
SIZE = None                # None: base на GPU, small на CPU | "small" | "base" | "large"
MAX_SIDE = None            # довша сторона кадру для щільних фіч (блоки 1–2); None: 896 на GPU, 672 на CPU
ATTN_MAX_SIDE = 672        # карта уваги [CLS] рахується на меншому кадрі: eager-увага тримає N×N усіх шарів
CROPS_PER_CLASS_SEQ = None # блок 3: кропів на (послідовність, клас); None: 40 на GPU, 15 на CPU
FRAMES_PER_SEQ = 12        # блок 4: кадрів на послідовність для карти архіву
BUDGETS = (12, 30, 60, 120, 300, 600)   # блок 3: скільки кропів «розмічає експерт»
SEEDS = (0, 1, 2)

import dino_features as D   # torch / transformers вантажаться лише в DinoEncoder

dev = D.pick_device(DEVICE)
on_gpu = dev.startswith("cuda")
SIZE = SIZE or ("base" if on_gpu else "small")
MAX_SIDE = MAX_SIDE or (896 if on_gpu else 672)
CROPS_PER_CLASS_SEQ = CROPS_PER_CLASS_SEQ or (40 if on_gpu else 15)

VISDRONE_DIR, CACHE_DIR = D.VISDRONE_DIR, Path("outputs/demo1")
if SMOKE:
    VISDRONE_DIR, CACHE_DIR = Path("outputs/smoke1/visdrone"), Path("outputs/smoke1/cache")
    MAX_SIDE, ATTN_MAX_SIDE, FRAMES_PER_SEQ, BUDGETS, SEEDS = 448, 336, 6, (4, 8), (0,)

os.environ.setdefault("HF_HOME", str(D.HF_HOME))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# Усі ваги вже завантажив tools/fetch_dino_models.py -> працюємо офлайн, без звернень до HF Hub.
_manifest = D.MODELS_DIR / "manifest.json"
if not SMOKE and _manifest.exists():
    _have = json.loads(_manifest.read_text())
    if all(f"{m}-{SIZE if SIZE in D.DINO_ZOO[m] else next(iter(D.DINO_ZOO[m]))}" in _have for m in MODELS):
        os.environ.setdefault("HF_HUB_OFFLINE", "1")

import numpy as np, pandas as pd
from PIL import Image
pd.set_option("display.precision", 3)
print("SMOKE" if SMOKE else "FULL", "| device:", dev, "| models:", MODELS, SIZE, "| max side:", MAX_SIDE,
      "| HF:", "offline" if os.environ.get("HF_HUB_OFFLINE") == "1" else "online")

## 0. Дані й моделі

VisDrone-VID: 80 послідовностей, train / val / test-dev. Кілька послідовностей train вирізано з одного вихідного відео
(`uav0000013_00000_v`, `uav0000013_01073_v`, … — спільний префікс `uavNNNNNNN`). Між сабсетами спільних відео немає.
Це знадобиться в блоці 4.

In [ ]:
if SMOKE:
    D.make_synthetic(VISDRONE_DIR)
elif not any(VISDRONE_DIR.glob("VisDrone2019-VID-*/sequences")):
    subprocess.run([sys.executable, str(D.REPO_ROOT / "tools" / "fetch_visdrone.py"),
                    "--splits", "train", "val", "test-dev"], check=True)

seqs = D.list_sequences(VISDRONE_DIR)
assert len(seqs), f"Послідовностей не знайдено в {VISDRONE_DIR}"
summary = seqs.groupby("split").agg(послідовностей=("seq", "size"), відео=("vid", "nunique"), кадрів=("n_frames", "sum"))
display(summary)
if not {"train", "val", "test-dev"} <= set(seqs.split):
    print("⚠ Для блоків 3–4 потрібні всі три сабсети: python tools/fetch_visdrone.py --splits train val test-dev")

In [ ]:
encs = D.load_encoders(MODELS, SIZE, device=dev, random_weights=SMOKE)
assert encs, "Жодна модель не завантажилась"
pd.DataFrame([dict(model=m, repo=e.repo, params_M=e.n_params / 1e6, patch=e.patch, registers=e.n_reg, dim=e.dim,
                   device=e.device, load_s=e.load_s) for m, e in encs.items()])

## 1. Що бачить модель без жодної мітки

Для кожної моделі чотири панелі:

* **PCA патч-фіч.** Три головні компоненти щільних фіч як RGB. Жодної мітки, а дороги, дахи, зелень і машини
  розходяться за кольором. Кольори між моделями не порівнюються: знак і порядок компонент довільні.
* **‖токен‖.** Норма кожного патч-токена. DINOv2 «паркує» глобальну інформацію в кількох патчах однорідного фону
  (асфальт, небо): вони мають аномально велику норму й псують щільні фічі. **Регістри** (Darcet et al., 2024)
  дають моделі окремі токени під це, і викиди зникають. DINOv3 має 4 регістри з коробки.
* **Увага [CLS].** Побічний ефект DINO зі слайду 26: карта уваги виділяє об'єкти без жодної маски.
* **Сітка.** DINOv2: patch 14, DINOv3: patch 16 (слайд 17). На тому самому кадрі в них різна кількість токенів,
  а ціль у 20 px — це 1–2 патчі.

In [ ]:
_pref = seqs[seqs.seq == "uav0000137_00458_v"]
r0 = _pref.iloc[0] if len(_pref) else seqs[seqs.split == seqs.split.iloc[-1]].iloc[0]
im0 = Image.open(D.frame_path(r0, r0.n_frames // 2))
D.show_feature_maps(im0, encs, MAX_SIDE, ATTN_MAX_SIDE, f"{r0.split}/{r0.seq} · {im0.width}×{im0.height}")

**Інтерактив.** Виберіть послідовність і кадр, змінюйте `max side`:

* 448 → 1344: кількість токенів росте квадратично, пам'ять уваги — у четвертому степені від сторони (O(N²), слайд 19).
* На 448 машини зливаються з дорогою, на 1344 стають окремими плямами. Це «рахуємо пікселі після ресайзу» зі слайду 22.
* Нічні послідовності (`val/uav0000117_02622_v`, `test-dev/uav0000119_02301_v`) і 4K-кадр `val/uav0000268_05773_v`
  показують, як поводяться фічі поза «денним вебовим» доменом.

In [ ]:
D.feature_maps_panel(seqs, encs, MAX_SIDE, ATTN_MAX_SIDE)

## 2. Клікни на ціль: подібність патчів

Беремо фічу одного патча (або середнє окілу 3×3) і рахуємо її косинус до **всіх** патчів кадру-запиту і кадрів галереї.
Запити — з val / test-dev, галерея — з train, усі з різних відео. Модель ці сцени не бачила, мітки не використовуються.

Шкала кольору в рядку однакова для всіх кадрів: від медіани до максимуму сирого cos (без самого клікнутого патча).
Максимальний cos на кадрі галереї — у заголовку.

Що спробувати:
* **машина** → машини світяться в чужих відео: one-shot-пошук без детектора, «новий клас = один клік»;
* **пішохід 15–20 px** → він займає ~1 патч, фіча змішана з асфальтом, сигнал слабкий (слайд 22);
* **зебра, дах, крона** → текстури теж шукаються, і саме вони дають хибні збіги;
* **v2 проти v3** → у DINOv3 карти чистіші й чіткіші на межах (Gram anchoring проти деградації щільних фіч).

In [ ]:
queries = D.pick_frames(seqs, ["val", "test-dev"], 8, seed=0)
gallery = D.pick_frames(seqs, ["train"], 5, exclude_vids=[q.split("/")[1].split("_")[0] for q, _ in queries], seed=1)
# стартова точка — центр найбільшої машини на першому запиті, щоб перший рендер був змістовним
_q = seqs[(seqs.split + "/" + seqs.seq) == queries[0][0]].iloc[0]
_b = D.largest_box(_q, int(0.5 * (_q.n_frames - 1)), cats=(4, 5, 6, 9))
start_xy = (_b[0] + _b[2] / 2, _b[1] + _b[3] / 2) if _b else None
panel = D.patch_similarity_panel(encs, queries, gallery, MAX_SIDE, start_xy)

## 3. «200 кадрів проти 2000»: скільки міток потрібно

Задача: класифікувати кропи об'єктів VisDrone (pedestrian / car / van / truck / bus / motor) за bbox з розмітки.
Фічі **заморожені**, навчається лише kNN або логістична регресія (linear probe), як у робочому рецепті зі слайду 30.

* **Пул для розмітки** — кропи train; **тест** — val + test-dev (інші відео).
* **Бюджет** — скільки кропів «розмітив експерт»: 12 … 600 проти всіх train-міток.
* **Випадковий добір** проти **кластерного**: k-means по ембедінгах з `budget` кластерами, розмічаємо кроп, найближчий до центру
  кожного кластера. Мітки на етапі добору невідомі, тож добір чесний.
* **Метрика** — macro-recall, а не accuracy (слайд 46): машин у VisDrone більшість, accuracy ховає провал на автобусах.

Перед запуском варто запитати аудиторію: скільки міток треба, щоб наблизитись до «всіх train-міток»?

In [ ]:
crops = D.extract_crops(seqs, cap_per_class_seq=CROPS_PER_CLASS_SEQ)
crops = D.materialize_crops(crops, CACHE_DIR / "crops")
classes = sorted(crops.cls.unique())
cls2id = {c: i for i, c in enumerate(classes)}
y = crops.cls.map(cls2id).to_numpy()
n_cls = len(classes)
tr = (crops.split == "train").to_numpy()
te = crops.split.isin(["val", "test-dev"]).to_numpy()
print(f"Кропів: {len(crops)} · train {tr.sum()} · test {te.sum()} · класів {n_cls}")
display(crops.groupby(["cls", "split"]).size().unstack(fill_value=0))
print("Коротка сторона, px вихідного кадру:", crops.short.describe()[["25%", "50%", "75%"]].to_dict())

In [ ]:
crop_embs, rows = {}, []
for m, e in encs.items():
    t0 = time.time()
    crop_embs[m] = D.embed_crops(e, crops, CACHE_DIR / "emb")
    rows.append(dict(model=m, vectors=len(crop_embs[m]), dim=crop_embs[m].shape[1], s=time.time() - t0,
                     img_per_s=e.img_per_s))
pd.DataFrame(rows)

In [ ]:
t0 = time.time()
R = D.label_efficiency(crop_embs, y, tr, te, n_cls, budgets=BUDGETS, seeds=SEEDS)
print(f"{len(R)} прогонів за {time.time() - t0:.0f} с")
D.plot_label_efficiency(R, n_cls)

tab = (R[R.clf == "kNN"].groupby(["model", "method", "budget"]).agg(macro_recall=("macro_recall", "mean"),
                                                                     класів_у_розмітці=("classes", "mean"))
       .unstack("model"))
display(tab.round(3))

**Що обговорити.**
* **Чи справді кластерний добір кращий?** Перевірте на графіку, а не на слайді. У прогоні DINOv2-base на VisDrone
  k-means виграв лише на найменшому бюджеті (~12 міток). Далі випадковий добір не гірший: найближчі до центрів кропи —
  «типові» машини й люди, а межу car / van / truck вони не покривають. Теза «200 таких кращі за 2000 випадкових» — гіпотеза,
  яку перевіряють на своєму домені (слайд 30).
* Колонка `класів_у_розмітці`: на 12 мітках у розмітку потрапляють не всі класи, і kNN рідкісний клас не передбачить ніколи.
* Linear probe на всіх мітках помітно сильніший за kNN. На малих бюджетах різниця менша.
* Найчесніше число для наради — бюджет, на якому крива виходить на плато відносно «всіх міток».

### Точність за розміром цілі

Ті самі фічі, kNN на всіх train-мітках, але macro-recall окремо за короткою стороною об'єкта у вихідному кадрі.
Кроп 15 px, розтягнутий до 224, — це розмита пляма: тут вирішує бюджет пікселів, а не модель (слайд 22).
Застереження: склад класів у кошиках різний (пішоходи переважно дрібні, автобуси великі), тож порівнюйте кошики обережно.

In [ ]:
S = D.recall_by_size(crop_embs, y, crops.short.to_numpy(), tr, te, n_cls)
piv = S.pivot(index="size", columns="model", values="macro_recall").reindex(S["size"].unique())
piv.insert(0, "n_test", S.drop_duplicates("size").set_index("size")["n"])
display(piv.round(3))
ax = piv.drop(columns="n_test").plot.bar(figsize=(8, 3.5), rot=0, ylabel="macro-recall", title="kNN за розміром цілі")
ax.axhline(1 / n_cls, color="grey", lw=0.8);

## 4. Архів як актив: карта сцен, дублікати, витік

CLS-ембедінг цілого кадру — компактний опис сцени. На ньому можна будувати карту архіву, шукати дублікати й аномалії
(слайд 26, «пошук дублікатів і аномалій в архіві»).

На карті ліворуч кольором позначено сабсет, праворуч підсвічено послідовності, вирізані з одного вихідного відео.
Вони лягають купками: для моделі це майже одна сцена.

In [ ]:
frames = D.sample_frames(seqs, FRAMES_PER_SEQ)
frame_embs = {m: D.embed_frames(e, frames, CACHE_DIR / "emb") for m, e in encs.items()}
print(f"Кадрів: {len(frames)} з {frames.seq.nunique()} послідовностей")
D.plot_scene_map(frame_embs, frames)

### Найсхожіші сцени з різних відео

Пари кадрів з **різних** `uavNNNNNNN` з найбільшим косинусом. Це кандидати в «ту саму локацію», тобто у витік між сабсетами.

На VisDrone топ — це **ті самі місця, зняті в інших польотах**: те саме перехрестя з білбордами (train/143 ↔ train/295),
баскетбольний комплекс (train/084 ↔ val/086 ↔ test-dev/088), вулиця з червоним фасадом (train/309 ↔ test-dev/306).
Отже, навіть офіційне розбиття VisDrone «за відео» має **витік за локацією** між train і val / test-dev.
Перевірка точковими ознаками (SIFT + RANSAC) цього не помітила: інший ракурс, висота й час доби. DINO — помітила.

Практичний висновок: пошук дублікатів по CLS — дешевий аудит датасету перед тим, як вірити метриці.
І навпаки: висока схожість кадрів ≠ та сама ціль (слайд 48), остаточно вирішують очі.

In [ ]:
ref = list(encs)[-1]   # найновіша з завантажених моделей
pairs = D.cross_video_pairs(frame_embs[ref], frames, top=6)
display(pairs[["cos", "a", "b", "cross_split"]])
D.show_pairs(pairs, f"{ref}: найсхожіші кадри з різних відео")

### Пастка: витік через розбиття train/test

Беремо лише train-кропи з блоку 3 і трьома способами відкладаємо 20 % на тест:

* **кроп випадково** — кропи одного треку й однієї сцени опиняються по обидва боки, і kNN «впізнає» сусіда;
* **за послідовністю** — чесніше, але послідовності одного відео (`uav0000013_*`, `uav0000072_*`, …) усе ще можуть розійтися;
* **за відео `uavNNNNNNN`** — так розбиті й офіційні train / val / test-dev. Але, як показав попередній крок,
  різні відео можуть знімати ту саму локацію, тож і це розбиття не ідеальне.

Для порівняння — та сама модель train → val + test-dev з блоку 3.

In [ ]:
L = D.leakage_experiment(crop_embs, crops, y, n_cls)
ref_rows = R[(R.method == "усі мітки") & (R.clf == "kNN")].assign(split="train → val + test-dev (офіційне)")
tab = (pd.concat([L, ref_rows[["model", "split", "macro_recall"]]])
       .groupby(["split", "model"]).macro_recall.agg(["mean", "std"]).unstack("model"))
order = ["кроп випадково", "за послідовністю", "за відео (uavNNNNNNN)", "train → val + test-dev (офіційне)"]
display(tab.reindex([o for o in order if o in tab.index]).round(3))

**Висновок блоку.** Різниця між першим і останніми рядками — це те, наскільки «красиве число» може брехати
(слайд 48: «невизначеність має бути видно»). У своєму архіві розбивайте за вильотом і **районом** зйомки, а не за кадром.

## Підсумок

1. **Структура без міток.** PCA і карти подібності показують: DINO розділяє дороги, дахи, машини й людей, не бачивши жодної мітки.
2. **DINOv3 vs DINOv2.** Регістри й Gram anchoring прибирають артефакти щільних фіч. Patch 16 замість 14 означає іншу сітку,
   а отже інший мінімальний розмір цілі.
3. **Розмітка.** Кластерний добір міток на заморожених фічах дає більше, ніж той самий бюджет випадково. Скільки саме —
   крива з блоку 3 на вашому домені.
4. **Чесні числа.** Без розбиття за вильотом і локацією метрика завищена, і DINO-ембедінги допомагають знайти такий витік
   навіть в офіційному датасеті. Дрібні цілі впираються в пікселі, а не в модель.

**Для самостійної роботи:** запустіть блоки 3–4 з `MODELS = ["dinov3", "dinov3sat"]`, `SIZE = "large"`. Чи допомагає
супутниковий DINOv3 на похилих кадрах дрона з висоти 30–100 м?